In [1]:
import json
import os
import random
import sys
from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from nnsight import CONFIG, LanguageModel
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import (
    error_detection,
    get_query_charac_oi,
    get_query_object_oi,
    get_reversed_sent_diff_state_counterfacts,
    get_reversed_sentence_counterfacts,
)

current_dir = os.getcwd()
sys.path.append(os.path.dirname(os.path.dirname(current_dir)))
from src import global_utils
from src.dataset import Sample, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(10)

CONFIG.APP.REMOTE_LOGGING = False
CONFIG.set_default_api_key(global_utils.load_env_var("NDIF_KEY"))
os.environ["HF_TOKEN"] = global_utils.load_env_var("HF_WRITE")

%load_ext autoreload

/disk/u/nikhil/belief_tracking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_characters = json.load(
    open(
        os.path.join(global_utils.DATA_DIR, "synthetic_entities", "characters.json"),
        "r",
    )
)
all_objects = json.load(
    open(
        os.path.join(global_utils.DATA_DIR, "synthetic_entities", "bottles.json"),
        "r",
    )
)
all_states = json.load(
    open(
        os.path.join(global_utils.DATA_DIR, "synthetic_entities", "drinks.json"),
        "r",
    )
)

print(f"#characters: {len(all_characters)}")
print(f"#objects: {len(all_objects)}")
print(f"#states: {len(all_states)}")

#characters: 103
#objects: 21
#states: 23


In [3]:
is_remote = False

if is_remote:
    model = LanguageModel("meta-llama/Llama-3.1-405B-Instruct")
else:
    model = LanguageModel(
        "meta-llama/Meta-Llama-3-70B-Instruct",
        device_map="auto",
        dtype=torch.float16,
        dispatch=True,
    )

Loading checkpoint shards: 100%|██████████| 30/30 [00:33<00:00,  1.13s/it]


In [47]:
n_samples = 1000
batch_size = 1

samples = []
for i in range(n_samples):
    characters = random.sample(all_characters, 2)
    objects = random.sample(all_objects, 2)
    states = random.sample(all_states, 2)

    sample = Sample(2, characters, objects, states)
    samples.append(sample)

dataset = Dataset(samples)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

In [48]:
next(iter(dataloader))

{'characters': [('Max',), ('Eric',)],
 'objects': [('horn',), ('bottle',)],
 'states': [('cocktail',), ('champagne',)],
 'story': ['Max and Eric are working in a busy restaurant. To complete an order, Max grabs an opaque horn and fills it with cocktail. Then Eric grabs another opaque bottle and fills it with champagne.'],
 'question': ['What does Max believe the bottle contains?'],
 'target': ['unknown'],
 'prompt': ["Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about the container in question, then predict 'unknown'. 6. Do not predict containe

In [49]:
errors = []

for bi, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    prompt = batch["prompt"][0]
    target = batch["target"][0]

    with torch.no_grad():
        with model.trace(prompt):
            pred = model.lm_head.output[0, -1].argmax(dim=-1).save()

    pred = model.tokenizer.decode(pred)
    # print(f"Index: {bi}, Pred: {pred}, Target: {target}")
    if pred.lower().strip() != target.lower().strip():
        errors.append(batch)

print(f"#errors: {len(errors)}")

100%|██████████| 1000/1000 [04:02<00:00,  4.12it/s]

#errors: 44


In [41]:
print(errors[0])

{'characters': [('Ivy',), ('Adam',)], 'objects': [('tun',), ('container',)], 'states': [('espresso',), ('port',)], 'story': ['Ivy and Adam are working in a busy restaurant. To complete an order, Ivy grabs an opaque tun and fills it with espresso. Then Adam grabs another opaque container and fills it with port.'], 'question': ['What does Ivy believe the container contains?'], 'target': ['unknown'], 'prompt': ["Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about the container in question, then predict 'unknown'. 6. Do not predict container or char

In [42]:
counterfactual_samples = []
for i in range(len(errors)):
    correct_sample = Sample(2, random.sample(all_characters, 2), random.sample(all_objects, 2), random.sample(all_states, 2))
    dataset = Dataset([correct_sample])
    correct_sample = dataset.__getitem__(0, set_character=errors[i]["character_idx"], set_container=errors[i]["object_idx"])
    prompt = correct_sample["prompt"]
    target = correct_sample["target"]
    with torch.no_grad():
        with model.trace(prompt):
            pred = model.lm_head.output[0, -1].argmax(dim=-1).save()

    pred = model.tokenizer.decode(pred)
    if pred.lower().strip() == target.lower().strip():
        counterfactual_samples.append({
            "prompt": prompt,
            "target": target,
        })

In [43]:
incorrect_sample = errors[0]
incorrect_prompt, incorrect_target = incorrect_sample["prompt"], incorrect_sample["target"]
correct_prompt, correct_target = correct_sample["prompt"], correct_sample["target"]

In [44]:
print(f"Correct Prompt: {correct_prompt} {correct_target}")
print("=" * 25)
print(f"Incorrect Prompt: {incorrect_prompt[0]} {incorrect_target[0]}")

Correct Prompt: Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about the container in question, then predict 'unknown'. 6. Do not predict container or character as the final output.

Story: Greg and Ivy are working in a busy restaurant. To complete an order, Greg grabs an opaque dispenser and fills it with rum. Then Ivy grabs another opaque pint and fills it with tea.
Question: What does Greg believe the pint contains?
Answer: unknown
Incorrect Prompt: Instruction: 1. Track the belief of each character as described in the story. 2. A character's 

In [ ]:
charac_indices = [131, 133, 146, 147, 158, 159]
object_indices = [150, 151, 162, 163]
state_indices = [155, 156, 167, 168]

with torch.no_grad():
    for layer_idx in range(0, model.config.num_hidden_layers, 4):
        corrupt_act_1, corrupt_act_2, corrupt_act_3, corrupt_act_4 = defaultdict(dict), defaultdict(dict), defaultdict(dict), defaultdict(dict)
        with model.trace(correct_prompt):
            for l in range(layer_idx, layer_idx + 1, 1):
                for t in [-5, -4]:
                    corrupt_act_2[l][t] = model.model.layers[l].mlp.output[:, t].save()
                    corrupt_act_1[l][t] = model.model.layers[l].self_attn.output[0][:, t].save()
                    corrupt_act_3[l][t] = model.model.layers[l].output[:, t].save()

        with model.trace(incorrect_prompt):
            for l in range(layer_idx, layer_idx + 1, 1):
                for t in [-5, -4]:
                    model.model.layers[l].output[:, t] = corrupt_act_3[l][t]

                # for t in [-5, -4]:
                #     model.model.layers[l].self_attn.output[0][:, t] = corrupt_act_1[l][t]

                # for t in [-5, -4]:
                #     model.model.layers[l].mlp.output[:, t] = corrupt_act_2[l][t]

            pred = model.lm_head.output[0, -1].argmax(dim=-1).save()

        print(f"Layer: {layer_idx}, New Pred: {model.tokenizer.decode([pred])}, Target: {incorrect_target[0]}")

        del corrupt_act_1, pred
        torch.cuda.empty_cache()


Layer: 0, New Pred:  unknown, Target: unknown
Layer: 4, New Pred:  unknown, Target: unknown
Layer: 8, New Pred:  unknown, Target: unknown
Layer: 12, New Pred:  espresso, Target: unknown
Layer: 16, New Pred:  espresso, Target: unknown
Layer: 20, New Pred:  espresso, Target: unknown
Layer: 24, New Pred:  espresso, Target: unknown
Layer: 28, New Pred:  espresso, Target: unknown
Layer: 32, New Pred:  espresso, Target: unknown
Layer: 36, New Pred:  espresso, Target: unknown
Layer: 40, New Pred:  espresso, Target: unknown
Layer: 44, New Pred:  espresso, Target: unknown
Layer: 48, New Pred:  espresso, Target: unknown
Layer: 52, New Pred:  espresso, Target: unknown
Layer: 56, New Pred:  espresso, Target: unknown
Layer: 60, New Pred:  espresso, Target: unknown
Layer: 64, New Pred:  espresso, Target: unknown
Layer: 68, New Pred:  espresso, Target: unknown
Layer: 72, New Pred:  espresso, Target: unknown
Layer: 76, New Pred:  espresso, Target: unknown
